## Суть хакатоны: мультитаргетовая классификация - обучить модель или модели, которые хорошо предсказывают значение 41 таргета. Это бинерная классификация, метрики: ROC-AUC. Что важно: это не "классификация", а больше ранжирование, ведь ROC-AUC оценивает в первую очередь способность модели отделять один класс от другого. 

## Что дано:
## train_main_features.parquet , размерность: 750к * 200
## train_extra_features.parquet , размерность: 750к * 2300
## train_target.parquet , размерность: 750к * 42

## Примечание: это задача часть хакатона Data Fest 2026 года. Я учавствовал в нём, но подключился поздно и данное решение уже отправить не смогу, но задача показалось интересной, и я решил её доделать. 

## Первая основная трудность: большой объём данных. Даже на kaggle - имея 30гб RAM, данные не влезают. 
## Вторая основная трудность: подбор параметров через optuna невозможен. Скажем, n_trail=30, таргетов 41, итого 1200 обучений модели. Берём среднее время обучения модели: 2 минуты и получаем: 2400 минут или 40  часов, и это на gpu. И это с одной моделью. 


## План ноутбука
## - Очистка данных
## - Обучение catboost на малой выборке по отдельным таргетам (один таргет - одна модель)
## - Выбор лучших фичей (топ-200 по важности "по мнению" моделей, другими словами, через model.feature_importance_)
## - Создание фичей путём комбинации топ-40 лучших.
## - Обучение catboost на новых данных по отдельным таргетам
## - Выбор топ-400 лучших фичей
## - Обучение catboost, xgboost, lightgbm, mlp (с attention) по всем таргетам на малой выборке (но больше чем та, на которой обучался первый и второй catboost)
## - Обучения ансамбля: стек из 4 моделей + лог рег над ними
## - Вывод результатов



In [ ]:
# !pip install -U xgboost

In [ ]:
import gc
import json
import re
from pathlib import Path
import os
import numpy as np
import pandas as pd
import polars as pl
import optuna
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import json

import gc
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import optuna

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

y = pd.read_parquet("/kaggle/input/datasets/kehhill/data-fest-2026-2-track/train_target.parquet")

train_mf_path = "/kaggle/input/datasets/kehhill/data-fest-2026-2-track/train_main_features.parquet"
train_ef_path = "/kaggle/input/datasets/kehhill/data-fest-2026-2-track/train_extra_features.parquet"



y_mini = y[y.columns.to_list()[1:16]]


# y = y[y.columns.to_list()[1:3]]
# y_mini = y.copy()

In [ ]:
pos_lim = 62500+5
col_lim = 125000+5


pos_lim_2 = 100000+5
col_lim_2 = 200000+5

## Фильтруем хорошие колонки

In [ ]:
import os
from pathlib import Path
import polars as pl
import pandas as pd
import gc
import numpy as np
def drop_shit_cols(data):
    good_cols = data.columns.to_numpy()
    good_cols = good_cols[data.isna().sum().to_numpy()/data.shape[0]<0.8]
    data = data[good_cols]
    const_cols = []
    for col in good_cols:   
        if data[col].nunique(dropna=True)==1:
            const_cols.append(col)
    good_cols = good_cols[~np.isin(good_cols,const_cols)]
    data = data[good_cols]


    for col in good_cols:   
        if 'cat' in col:    
            data[col] = data[col].fillna('unk')
        elif 'num' in col:
            data[col] = data[col].fillna(data[col].median())
    return data



train_data = pd.read_parquet(train_mf_path).iloc[:, 1:]
train_data = drop_shit_cols(train_data)

new_train_mf_path = '/kaggle/working/train_main.parquet'
train_data.to_parquet(new_train_mf_path)
good_cols = train_data.columns.to_list()

big_d_cols = list(pl.read_parquet_schema(train_ef_path))[1:]


n_in_part = 300
for i in range(len(big_d_cols)//n_in_part+1):
    print(f"{i+1}/{len(big_d_cols)//n_in_part+1}")
    transform_data = pd.read_parquet(train_ef_path,
                                  columns=big_d_cols[i*n_in_part:i*n_in_part+n_in_part])
    transform_data = drop_shit_cols(transform_data)
    good_cols.extend(transform_data.columns.to_list())
    transform_data.to_parquet(f'train_extra_features_{i}.parquet')





## Собираем наборы индексы для формирование информативных малых выборок

In [ ]:
import numpy as np

rng = np.random.default_rng(2)

indeex_for_y_cols_train = pd.DataFrame()
indeex_for_y_cols_test = pd.DataFrame()
# indeex_for_y_cols = pd.DataFrame()
test_size=0.2


for y_col in y.columns.to_list():
    y_targ = np.array(y[y_col])

    pos_idx = np.flatnonzero(y_targ == 1)
    neg_idx = np.flatnonzero(y_targ == 0)

    rng.shuffle(pos_idx)
    rng.shuffle(neg_idx)

    if len(pos_idx) > pos_lim:
        pos_idx = pos_idx[:pos_lim]

    need_neg = col_lim - len(pos_idx)

    if len(neg_idx) > need_neg:
        neg_idx = neg_idx[:need_neg]

    pos_split = int(len(pos_idx) * (1 - test_size))
    neg_split = int(len(neg_idx) * (1 - test_size))

    idx_train = np.concatenate([
        pos_idx[:pos_split],
        neg_idx[:neg_split]
    ])

    idx_test = np.concatenate([
        pos_idx[pos_split:],
        neg_idx[neg_split:]
    ])

    rng.shuffle(idx_train)
    rng.shuffle(idx_test)

    idx_train = idx_train[:int((col_lim - 5) * (1 - test_size))]
    idx_test = idx_test[:int((col_lim - 5) * test_size)]

    indeex_for_y_cols_train[y_col] = idx_train
    indeex_for_y_cols_test[y_col] = idx_test
    

In [ ]:
md_features = train_data.columns.to_list()
ed_features = [c for c in good_cols if c not in md_features]

In [ ]:

# тест ram-а, сколько умещается
first_open=True

y_col = y.columns.to_list()[1]


X_train = train_data.iloc[indeex_for_y_cols_train[y_col]]
X_val = train_data.iloc[indeex_for_y_cols_test[y_col]]

for path_extra in [p for p in os.listdir('/kaggle/working/') if 'train_extra_features' in p]:
    X_train = pd.concat([X_train, pd.read_parquet(path_extra).iloc[indeex_for_y_cols_train[y_col]]],axis=1)
    X_val = pd.concat([X_val, pd.read_parquet(path_extra).iloc[indeex_for_y_cols_test[y_col]]],axis=1)


## Обучаем базовый catboost

#### Это не итоговая модель, а скорее промежуточная, обучаем не на всех данных!

In [ ]:
def create_cb(scale_pos_weight):
        
        # Базовые фиксированные параметры
        base_params = {
                "loss_function": "Logloss",
                "eval_metric": "AUC",
                "iterations": 1400,
                "random_seed": 42,
                "verbose": 0,
                "thread_count": 1,
                "scale_pos_weight": scale_pos_weight,
                "task_type": "GPU",
                "devices": "0",
                "od_type": "Iter",
                "od_wait": 200,
            }
        best_params = {'learning_rate': 0.03, 'depth': 6, 
                           'l2_leaf_reg': 4.0, 'bagging_temperature': 0.2, 
                           'random_strength': 0.4, 'border_count': 64}
        
        
        final_params = {
                **base_params,
                **best_params,
                "verbose": 100,
            }

        model = CatBoostClassifier(**final_params)
        return model





MODELS_DIR = Path("/kaggle/working/catboost_baseline")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

macro_roc_auc = []
trained_models = {}
models_meta = {}


first_col = True
n = 0

for y_col in y_mini.columns.to_list():
    n+=1
    print(f'Start {n}/41 part')
    print(f"Start training model for {y_col}!")
    print("=" * 30)
    
    X_train = train_data.iloc[indeex_for_y_cols_train[y_col]]
    X_val = train_data.iloc[indeex_for_y_cols_test[y_col]]
    
    for path_extra in [p for p in os.listdir('/kaggle/working/') if 'train_extra_features' in p]:
        X_train = pd.concat([X_train, pd.read_parquet(path_extra).iloc[indeex_for_y_cols_train[y_col]]],axis=1)
        X_val = pd.concat([X_val, pd.read_parquet(path_extra).iloc[indeex_for_y_cols_test[y_col]]],axis=1)

        
    y_train = y[y_col].astype(int).iloc[indeex_for_y_cols_train[y_col]]
    y_val = y[y_col].astype(int).iloc[indeex_for_y_cols_test[y_col]]

    n_pos = int((y_train == 1).sum())
    n_neg = int((y_train == 0).sum())

    # Берём только те категориальные колонки, которые реально есть в текущем датасете
    cat_cols = [col for col in X_train.columns.to_list() if 'cat' in col]

    # CatBoost не принимает float внутри категориальных колонок
    for col in cat_cols:
        X_train[col] = X_train[col].astype(str)
        X_val[col] = X_val[col].astype(str)
    

    scale_pos_weight = (n_neg / n_pos)**0.6
    model = create_cb(scale_pos_weight)

    model.fit(
        X_train,
        y_train,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    train_probs = model.predict_proba(X_train)[:, 1]
    val_probs = model.predict_proba(X_val)[:, 1]

    train_roc_auc = roc_auc_score(y_train, train_probs) if y_train.nunique() > 1 else 0.5
    valid_roc_auc = roc_auc_score(y_val, val_probs) if y_val.nunique() > 1 else 0.5

    trained_models[y_col] = model
    macro_roc_auc.append(valid_roc_auc)

    # Сохраняем модель
    model_filename = f"catboost_{y_col}.cbm"
    model_path = MODELS_DIR / model_filename
    model.save_model(str(model_path))

    # Сохраняем метаданные
    models_meta[y_col] = {
        "model_path": str(model_path),
        "train_roc_auc": float(train_roc_auc),
        "valid_roc_auc": float(valid_roc_auc)
    }

    print(
        f"Result for {y_col}:\n"
        f"   train_roc_auc={train_roc_auc:.6f}\n"
        f"   valid_roc_auc={valid_roc_auc:.6f}\n"
        f"   saved_to={model_path}"
    )
    print("\n" + "=" * 30 + "\n")

    del X_train, y_train, X_val, y_val, model, train_probs, val_probs
    gc.collect()


    # Общий metadata-файл
    meta_path = MODELS_DIR / "models_meta.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(models_meta, f, ensure_ascii=False, indent=2)
    
print("Macro roc-auc:", sum(macro_roc_auc) / len(macro_roc_auc))
print(f"Metadata saved to: {meta_path}")

## Достаём из бейслайна информацию о важности фичей

In [ ]:
import json

top_cols_for_targs = pd.DataFrame()


models = {}

fut_imps = []
for y_col in y_mini.columns.to_list():
    model = CatBoostClassifier()
    model.load_model(os.path.join(MODELS_DIR, f'catboost_{y_col}.cbm'))
    fut_imp = model.get_feature_importance()
    col_names = np.array(model.feature_names_)[np.argsort(fut_imp)[::-1]]
    top_cols_for_targs[y_col] = list(col_names)+['skip' for i in range(len(good_cols)-len(col_names))]
    top_cols_for_targs[y_col+'_val'] = list(fut_imp)+[0 for i in range(len(good_cols)-len(col_names))]

col_and_vals = pd.DataFrame()
col_and_vals['cols'] = [col for y_col in y_mini.columns.to_list() for col in top_cols_for_targs[y_col]]
col_and_vals['vals'] = [col for y_col in y_mini.columns.to_list() for col in top_cols_for_targs[y_col+'_val']]
col_and_vals = col_and_vals.groupby('cols').sum().reset_index()
col_and_vals = col_and_vals.sort_values(by='vals', ascending=False)
col_and_vals.to_parquet('columns_rating.parquet')



In [ ]:
top_40_cols = col_and_vals['cols'].to_list()[:40]
top_cols = col_and_vals['cols'].to_list()[:200]

### Выбираем топ лучших колонок 200 лучших колонок
### А да для топ-40 создаём датасет из их комбинаций

In [ ]:
md_features = [c for c in train_data.columns.to_list() if c in top_cols]
ed_features = [c for c in good_cols if c not in md_features and c in top_cols]

In [ ]:
train_data = pd.read_parquet(new_train_mf_path, columns=md_features)

In [ ]:


for te_path in [p for p in os.listdir("/kaggle/working/") if 'train_extra_features' in p]:
    te_full_path = os.path.join("/kaggle/working/",te_path)
    te_cols = pl.scan_parquet(te_full_path).collect_schema().names()
    te_cols = [c for c in te_cols if c in top_cols]
    train_data = pd.concat([train_data, \
                            pd.read_parquet(te_full_path, columns=te_cols)],axis=1)


In [ ]:
def create_comd_features(data, top_cols=top_40_cols):
    top_cols = [c for c in top_cols if c in data.columns.to_list()]
    cat_cols = [c for c in top_cols if 'cat' in c]
    num_cols = [c for c in top_cols if 'num' in c]
    
    for cat_col in cat_cols: 
        for num_col in num_cols:
            data[f"{cat_col}_mean_{num_col}"] = data.groupby(cat_col)[num_col].transform("mean")
            data[f"{cat_col}_max_{num_col}"] = data.groupby(cat_col)[num_col].transform("max")
            data[f"{cat_col}_min_{num_col}"] = data.groupby(cat_col)[num_col].transform("min") 
            data[f"{cat_col}_q25_{num_col}"] = data.groupby(cat_col)[num_col].transform(lambda x: x.quantile(0.25))
            data[f"{cat_col}_q50_{num_col}"] = data.groupby(cat_col)[num_col].transform(lambda x: x.quantile(0.50))
            data[f"{cat_col}_q75_{num_col}"] = data.groupby(cat_col)[num_col].transform(lambda x: x.quantile(0.75)) 
            data[f"{cat_col}_std_{num_col}"] = data.groupby(cat_col)[num_col].transform("std") 
    i = 0
    for num_col in num_cols:
        for num_col_2 in num_cols[i:]:
            data[f"{num_col}_diff_{num_col_2}"] = data[num_col]-data[num_col_2]
            data[f"{num_col}_ratio_{num_col_2}"] = data[num_col]/(data[num_col_2]+1e-5)
        i+=1
    return data


# train_data = pd.read_parquet("/kaggle/input/odsods/data/train_data.parquet")
print('Сейчас колонок: ', train_data.shape[1])
n=0
i = 0
for c in [i for i in top_40_cols if 'num' in i]:
    for c2 in [i for i in top_40_cols if 'cat' in i]:
        n+=7
    for c2 in [i for i in top_40_cols if 'num' in i][i:]:
        n+=2
    i+=1
print(f'Прибавиться колонок +{n}')
print(f'Итого будет {n+train_data.shape[1]} колонок')


train_extra_path = "/kaggle/working/train_data_best_features.parquet"
train_data = create_comd_features(train_data)

train_data.to_parquet(train_extra_path)
# del train_data; gc.collect()

In [ ]:
feature_names = pl.scan_parquet(train_extra_path).collect_schema().names()
len(feature_names)

## Снова обучаем catboost

In [ ]:
MODELS_DIR = Path("/kaggle/working/catboost_comb")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

macro_roc_auc = []
trained_models = {}
models_meta = {}


first_col = True
n = 0

for y_col in y_mini.columns.to_list():
    n+=1
    print(f'Start {n}/41 part')
    print(f"Start training model for {y_col}!")
    print("=" * 30)
    
 
    X_train = train_data.iloc[indeex_for_y_cols_train[y_col]]
    X_val = train_data.iloc[indeex_for_y_cols_test[y_col]]
    n_in_part = 300
    first_open = True
    for i in range(len(ed_features)//n_in_part+1):
        col_n = feature_names[i*n_in_part: i*n_in_part+n_in_part]
        if first_open:
            X_train = pd.read_parquet(train_extra_path, columns=col_n).iloc[indeex_for_y_cols_train[y_col]]
            X_val = pd.read_parquet(train_extra_path, columns=col_n).iloc[indeex_for_y_cols_test[y_col]]
        else:
            X_train = pd.concat([X_train, pd.read_parquet(train_extra_path, columns=col_n).iloc[indeex_for_y_cols_train[y_col]]],axis=1)
            X_val = pd.concat([X_val, pd.read_parquet(train_extra_path, columns=col_n).iloc[indeex_for_y_cols_test[y_col]]],axis=1)
        
        
    y_train = y[y_col].astype(int).iloc[indeex_for_y_cols_train[y_col]]
    y_val = y[y_col].astype(int).iloc[indeex_for_y_cols_test[y_col]]

    n_pos = int((y_train == 1).sum())
    n_neg = int((y_train == 0).sum())

    # Берём только те категориальные колонки, которые реально есть в текущем датасете
    cat_cols = [col for col in X_train.columns.to_list() if 'cat' in col]

    # CatBoost не принимает float внутри категориальных колонок
    for col in cat_cols:
        X_train[col] = X_train[col].astype(str)
        X_val[col] = X_val[col].astype(str)
    

    scale_pos_weight = (n_neg / n_pos)**0.6
    model = create_cb(scale_pos_weight)

    model.fit(
        X_train,
        y_train,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    train_probs = model.predict_proba(X_train)[:, 1]
    val_probs = model.predict_proba(X_val)[:, 1]

    train_roc_auc = roc_auc_score(y_train, train_probs) if y_train.nunique() > 1 else 0.5
    valid_roc_auc = roc_auc_score(y_val, val_probs) if y_val.nunique() > 1 else 0.5

    trained_models[y_col] = model
    macro_roc_auc.append(valid_roc_auc)

    # Сохраняем модель
    model_filename = f"catboost_{y_col}.cbm"
    model_path = MODELS_DIR / model_filename
    model.save_model(str(model_path))

    # Сохраняем метаданные
    models_meta[y_col] = {
        "model_path": str(model_path),
        "train_roc_auc": float(train_roc_auc),
        "valid_roc_auc": float(valid_roc_auc)
    }

    print(
        f"Result for {y_col}:\n"
        f"   train_roc_auc={train_roc_auc:.6f}\n"
        f"   valid_roc_auc={valid_roc_auc:.6f}\n"
        f"   saved_to={model_path}"
    )
    print("\n" + "=" * 30 + "\n")

    del X_train, y_train, X_val, y_val, model, train_probs, val_probs
    gc.collect()


    # Общий metadata-файл
    meta_path = MODELS_DIR / "models_meta.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(models_meta, f, ensure_ascii=False, indent=2)
    
print("Macro roc-auc:", sum(macro_roc_auc) / len(macro_roc_auc))
print(f"Metadata saved to: {meta_path}")

## Снова достаём из catboost-а лучшие фичи и получаем итоговый датасет

In [ ]:
MODELS_DIR

In [ ]:
import json

top_cols_for_targs = pd.DataFrame()


models = {}

fut_imps = []
for y_col in y_mini.columns.to_list():
    model = CatBoostClassifier()
    model.load_model(os.path.join(MODELS_DIR, f'catboost_{y_col}.cbm'))
    fut_imp = model.get_feature_importance()
    col_names = np.array(model.feature_names_)[np.argsort(fut_imp)[::-1]]
    top_cols_for_targs[y_col] = list(col_names)+['skip' for i in range(len(good_cols)-len(col_names))]
    top_cols_for_targs[y_col+'_val'] = list(fut_imp)+[0 for i in range(len(good_cols)-len(col_names))]

col_and_vals = pd.DataFrame()
col_and_vals['cols'] = [col for y_col in y_mini.columns.to_list() for col in top_cols_for_targs[y_col]]
col_and_vals['vals'] = [col for y_col in y_mini.columns.to_list() for col in top_cols_for_targs[y_col+'_val']]
col_and_vals = col_and_vals.groupby('cols').sum().reset_index()
col_and_vals = col_and_vals.sort_values(by='vals', ascending=False)
col_and_vals = col_and_vals[~(col_and_vals['cols']=="skip")]
col_and_vals.to_parquet('columns_rating_final.parquet')



### Берём 400 лучших колонок

In [ ]:
top_cols = col_and_vals['cols'].to_list()[:400]

In [ ]:
final_train_data = pd.read_parquet(train_extra_path, columns=top_cols)

## Отлично, мы получили хорошие, чистые данные. Теперь уже можно заняться полноценным обучением моделей.

In [ ]:
def create_xgb(scale_pos_weight):
    final_params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "n_estimators": 1400,
        "learning_rate": 0.02,
        "random_state": 42,
        "verbosity": 0,
        "n_jobs": 1,
        "scale_pos_weight": scale_pos_weight,

        "device": "cuda",
        "tree_method": "hist",

        "max_depth": 5,
        "reg_lambda": 3.0,
        "reg_alpha": 0.2,
        "min_child_weight": 10.0,
        "gamma": 0.1,
        "max_bin": 64,
        "subsample": 0.85,
        "colsample_bytree": 0.85,

        "early_stopping_rounds": 200,
    }

    return XGBClassifier(**final_params)

from lightgbm import LGBMClassifier


def create_lgbm(scale_pos_weight):

    base_params = {
        "objective": "binary",
        "metric": "auc",

        "n_estimators": 1400,
        "learning_rate": 0.02,

        "random_state": 42,
        "verbosity": -1,
        "n_jobs": 1,
        "scale_pos_weight": scale_pos_weight,

        # GPU
        "device": "gpu",
        "gpu_platform_id": 0,
        "gpu_device_id": 0,

        "boosting_type": "gbdt",
    }

    best_params = {
        # CatBoost depth=6
        # Для LightGBM не надо делать слишком много листьев
        "max_depth": 6,
        "num_leaves": 31,

        # регуляризация
        "reg_lambda": 5.0,
        "reg_alpha": 0.2,

        # аналог border_count=64
        "max_bin": 64,

        # аналог bagging_temperature/random_strength
        "feature_fraction": 0.85,
        "bagging_fraction": 0.85,
        "bagging_freq": 1,

        # защита от мелких переобученных листьев
        "min_child_samples": 60,
        "min_child_weight": 1e-2,
    }

    final_params = {
        **base_params,
        **best_params,
    }

    model = LGBMClassifier(**final_params)
    return model

## Куда без нейронок. MLP + attention

In [ ]:
import torch
import torch.nn as nn  # This imports the nn module
import torch.nn.functional as F

class TabularMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.attn = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.Sigmoid()
        )

        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.SiLU(),
            nn.Dropout(0.25),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(0.20),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.15),

            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.Dropout(0.10),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x * self.attn(x)
        return self.net(x).squeeze(1)


def train_mlp(
    X_train,
    y_train,
    X_val,
    y_val,
    scale_pos_weight,
    epochs=50,
    batch_size=4096,
    patience=7,
    lr=1e-3,
    weight_decay=1e-4
):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

    X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val_np = y_val.values

    train_loader = DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor),
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    model = TabularMLP(X_train.shape[1]).to(device)

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([scale_pos_weight], dtype=torch.float32, device=device)
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=3
    )

    best_auc = -1
    best_state = None
    bad_epochs = 0

    for epoch in range(1, epochs + 1):
        model.train()

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()

        model.eval()

        with torch.no_grad():
            val_logits = model(X_val_tensor)
            val_probs = torch.sigmoid(val_logits).cpu().numpy()

        valid_roc_auc = roc_auc_score(y_val_np, val_probs) if y_val.nunique() > 1 else 0.5

        scheduler.step(valid_roc_auc)

        print(f"Epoch {epoch}: valid_roc_auc={valid_roc_auc:.6f}")

        if valid_roc_auc > best_auc:
            best_auc = valid_roc_auc
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print(f"Early stopping on epoch {epoch}")
            break

    model.load_state_dict(best_state)

    return model

## Колонок теперь меньше: увеличим размер малой выборки

In [ ]:
import numpy as np


rng = np.random.default_rng(2)

indeex_for_y_cols_train = pd.DataFrame()
indeex_for_y_cols_test = pd.DataFrame()
# indeex_for_y_cols = pd.DataFrame()
test_size=0.2


for y_col in y.columns.to_list():
    y_targ = np.array(y[y_col])

    pos_idx = np.flatnonzero(y_targ == 1)
    neg_idx = np.flatnonzero(y_targ == 0)

    rng.shuffle(pos_idx)
    rng.shuffle(neg_idx)

    if len(pos_idx) > pos_lim_2:
        pos_idx = pos_idx[:pos_lim_2]

    need_neg = col_lim_2 - len(pos_idx)

    if len(neg_idx) > need_neg:
        neg_idx = neg_idx[:need_neg]

    pos_split = int(len(pos_idx) * (1 - test_size))
    neg_split = int(len(neg_idx) * (1 - test_size))

    idx_train = np.concatenate([
        pos_idx[:pos_split],
        neg_idx[:neg_split]
    ])

    idx_test = np.concatenate([
        pos_idx[pos_split:],
        neg_idx[neg_split:]
    ])

    rng.shuffle(idx_train)
    rng.shuffle(idx_test)

    idx_train = idx_train[:int((col_lim_2 - 5) * (1 - test_size))]
    idx_test = idx_test[:int((col_lim_2 - 5) * test_size)]

    indeex_for_y_cols_train[y_col] = idx_train
    indeex_for_y_cols_test[y_col] = idx_test
    

## Код обучения

In [ ]:
import xgboost as xgb
print(xgb.__version__)

In [ ]:
from pathlib import Path
import json
import gc
import joblib
import pandas as pd
import numpy as np
# базовое
import gc
import json
import copy
from pathlib import Path

# работа с данными
import pandas as pd
import numpy as np

# sklearn
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from xgboost.callback import EarlyStopping

# бустинги
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier

# сохранение моделей
import joblib

# pytorch (MLP)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder


MODELS_DIR = Path("/kaggle/working/models")
PREDS_DIR = Path("/kaggle/working/predictions")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
PREDS_DIR.mkdir(parents=True, exist_ok=True)

macro_roc_auc = []
models_meta = {}

n = 0

macro_roc_auc = {k: [] for k in ['catboost','xgb','lgbm', 'mlp']}

for y_col in y.columns.to_list()[1:]:
    n += 1
    print(f'Start {n}/41 part')
    print(f"Start training model for {y_col}!")
    print("=" * 30)

    X_train = pd.read_parquet(train_extra_path, columns=top_cols).iloc[indeex_for_y_cols_train[y_col]]
    X_val   = pd.read_parquet(train_extra_path, columns=top_cols).iloc[indeex_for_y_cols_test[y_col]]

    y_train = y[y_col].astype(int).iloc[indeex_for_y_cols_train[y_col]]
    y_val   = y[y_col].astype(int).iloc[indeex_for_y_cols_test[y_col]]

    n_pos = int((y_train == 1).sum())
    n_neg = int((y_train == 0).sum())

    cat_cols = [col for col in X_train.columns if 'cat' in col]
    feature_names = list(X_train.columns)

    for col in cat_cols:
        X_train[col] = X_train[col].astype(str)
        X_val[col]   = X_val[col].astype(str)

    scale_pos_weight = (n_neg / n_pos) ** 0.6 if n_pos > 0 else 1.0

    # =========================
    # CATBOOST
    # =========================
    model = create_cb(scale_pos_weight)

    model.fit(
        X_train,
        y_train,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    train_probs = model.predict_proba(X_train)[:, 1]
    val_probs   = model.predict_proba(X_val)[:, 1]

    train_roc_auc = roc_auc_score(y_train, train_probs) if y_train.nunique() > 1 else 0.5
    valid_roc_auc = roc_auc_score(y_val, val_probs) if y_val.nunique() > 1 else 0.5

    model_dir = MODELS_DIR / "catboost"
    pred_model_dir = PREDS_DIR / "catboost"

    model_dir.mkdir(exist_ok=True)
    pred_model_dir.mkdir(exist_ok=True)

    model_path = model_dir / f"{y_col}.cbm"

    model.save_model(str(model_path))

    pd.DataFrame({"pred": train_probs}).to_parquet(pred_model_dir / f"train_catboost_{y_col}.parquet")
    pd.DataFrame({"pred": val_probs}).to_parquet(pred_model_dir / f"val_catboost_{y_col}.parquet")

    meta_path = model_dir / "meta_model.json"

    if meta_path.exists():
        with open(meta_path, "r", encoding="utf-8") as f:
            meta_model = json.load(f)
    else:
        meta_model = {}

    meta_model[y_col] = {
        "model_path": str(model_path),
        "train_roc_auc": float(train_roc_auc),
        "valid_roc_auc": float(valid_roc_auc),
        "scale_pos_weight": float(scale_pos_weight),
        "cat_cols": cat_cols,
        "feature_names": feature_names,
    }

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta_model, f, ensure_ascii=False, indent=2)

    macro_roc_auc["catboost"].append(valid_roc_auc)

    print(f"CatBoost done: {valid_roc_auc:.6f}")

    # =========================
    # LIGHTGBM
    # =========================
    for col in [i for i in X_train.columns.to_list() if 'cat' in i]:
        X_train[col] = X_train[col].astype('float')
        X_val[col] = X_val[col].astype('float')
    model = create_lgbm(scale_pos_weight)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        categorical_feature=cat_cols,
        callbacks=[
            early_stopping(stopping_rounds=50),
            log_evaluation(period=100),
        ],
    )


    train_probs = model.predict_proba(X_train)[:, 1]
    val_probs   = model.predict_proba(X_val)[:, 1]

    train_roc_auc = roc_auc_score(y_train, train_probs) if y_train.nunique() > 1 else 0.5
    valid_roc_auc = roc_auc_score(y_val, val_probs) if y_val.nunique() > 1 else 0.5

    model_dir = MODELS_DIR / "lgbm"
    pred_model_dir = PREDS_DIR / "lgbm"

    model_dir.mkdir(exist_ok=True)
    pred_model_dir.mkdir(exist_ok=True)

    model_path = model_dir / f"{y_col}.pkl"

    joblib.dump(model, model_path)

    pd.DataFrame({"pred": train_probs}).to_parquet(pred_model_dir / f"train_lgbm_{y_col}.parquet")
    pd.DataFrame({"pred": val_probs}).to_parquet(pred_model_dir / f"val_lgbm_{y_col}.parquet")

    meta_path = model_dir / "meta_model.json"

    if meta_path.exists():
        with open(meta_path, "r", encoding="utf-8") as f:
            meta_model = json.load(f)
    else:
        meta_model = {}

    meta_model[y_col] = {
        "model_path": str(model_path),
        "train_roc_auc": float(train_roc_auc),
        "valid_roc_auc": float(valid_roc_auc),
        "scale_pos_weight": float(scale_pos_weight),
        "cat_cols": cat_cols,
        "feature_names": feature_names,
    }

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta_model, f, ensure_ascii=False, indent=2)

    macro_roc_auc["lgbm"].append(valid_roc_auc)

    print(f"LightGBM done: {valid_roc_auc:.6f}")

    # =========================
    # ONE HOT
    # =========================
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

    cat_cols = [col for col in X_train.columns.to_list() if 'cat' in col]
    num_cols = [col for col in X_train.columns.to_list() if 'num' in col]
    
    X_train_cat = ohe.fit_transform(X_train[cat_cols])
    X_val_cat   = ohe.transform(X_val[cat_cols])
    ohe_feature_names = list(ohe.get_feature_names_out(cat_cols))
    X_train_cat = pd.DataFrame(X_train_cat, columns=ohe_feature_names, index=X_train.index)
    X_val_cat   = pd.DataFrame(X_val_cat, columns=ohe_feature_names, index=X_val.index)
    X_train = pd.concat([X_train[num_cols], X_train_cat],axis=1)
    X_val = pd.concat([X_val[num_cols], X_val_cat],axis=1)
    del X_train_cat, X_val_cat; gc.collect()
    

    # =========================
    # XGBOOST
    # =========================
    model = create_xgb(scale_pos_weight)


    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    train_probs = model.predict_proba(X_train)[:, 1]
    val_probs   = model.predict_proba(X_val)[:, 1]

    train_roc_auc = roc_auc_score(y_train, train_probs) if y_train.nunique() > 1 else 0.5
    valid_roc_auc = roc_auc_score(y_val, val_probs) if y_val.nunique() > 1 else 0.5

    model_dir = MODELS_DIR / "xgb"
    pred_model_dir = PREDS_DIR / "xgb"

    model_dir.mkdir(exist_ok=True)
    pred_model_dir.mkdir(exist_ok=True)

    model_path = model_dir / f"{y_col}.json"

    model.save_model(str(model_path))

    pd.DataFrame({"pred": train_probs}).to_parquet(pred_model_dir / f"train_xgb_{y_col}.parquet")
    pd.DataFrame({"pred": val_probs}).to_parquet(pred_model_dir / f"val_xgb_{y_col}.parquet")

    meta_path = model_dir / "meta_model.json"

    if meta_path.exists():
        with open(meta_path, "r", encoding="utf-8") as f:
            meta_model = json.load(f)
    else:
        meta_model = {}

    meta_model[y_col] = {
        "model_path": str(model_path),
        "train_roc_auc": float(train_roc_auc),
        "valid_roc_auc": float(valid_roc_auc),
        "scale_pos_weight": float(scale_pos_weight),
        "feature_names": feature_names,
        "ohe_feature_names": ohe_feature_names,
    }

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta_model, f, ensure_ascii=False, indent=2)

    macro_roc_auc["xgb"].append(valid_roc_auc)

    print(f"XGBoost done: {valid_roc_auc:.6f}")

    # =========================
    # MLP
    # =========================

    
    
    
    model = train_mlp(
        X_train=X_train.values,
        y_train=y_train,
        X_val=X_val.values,
        y_val=y_val,
        scale_pos_weight=scale_pos_weight,
        epochs=50,
        batch_size=512,
        patience=7,
        lr=1e-3,
        weight_decay=1e-4
    )
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    model.eval()
    
    with torch.inference_mode():
        train_probs = torch.sigmoid(
            model(torch.tensor(X_train.values, dtype=torch.float32).to(device))
        ).detach().cpu().numpy()
    
        val_probs = torch.sigmoid(
            model(torch.tensor(X_val.values, dtype=torch.float32).to(device))
        ).detach().cpu().numpy()
    
    train_roc_auc = roc_auc_score(y_train, train_probs) if y_train.nunique() > 1 else 0.5
    valid_roc_auc = roc_auc_score(y_val, val_probs) if y_val.nunique() > 1 else 0.5
    
    model_dir = MODELS_DIR / "mlp"
    pred_model_dir = PREDS_DIR / "mlp"
    
    model_dir.mkdir(exist_ok=True)
    pred_model_dir.mkdir(exist_ok=True)
    
    model_path = model_dir / f"{y_col}.pt"
    
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "input_dim": X_train.shape[1],
            "scale_pos_weight": float(scale_pos_weight),
            "model_class": "TabularMLP",
        },
        model_path
    )
    
    pd.DataFrame({"pred": train_probs}).to_parquet(
        pred_model_dir / f"train_mlp_{y_col}.parquet"
    )
    
    pd.DataFrame({"pred": val_probs}).to_parquet(
        pred_model_dir / f"val_mlp_{y_col}.parquet"
    )
    
    meta_path = model_dir / "meta_model.json"
    
    if meta_path.exists():
        with open(meta_path, "r", encoding="utf-8") as f:
            meta_model = json.load(f)
    else:
        meta_model = {}
    
    meta_model[y_col] = {
        "model_path": str(model_path),
        "train_roc_auc": float(train_roc_auc),
        "valid_roc_auc": float(valid_roc_auc),
        "scale_pos_weight": float(scale_pos_weight),
        "feature_names": feature_names,
        "ohe_feature_names": ohe_feature_names,
        "input_dim": int(X_train.shape[1]),
    }
    
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta_model, f, ensure_ascii=False, indent=2)
    
    macro_roc_auc["mlp"].append(valid_roc_auc)
    
    print(f"MLP done: {valid_roc_auc:.6f}")
    
    del model
    torch.cuda.empty_cache()
    

    print(f"MLP done: {valid_roc_auc:.6f}")
    

    print("=" * 30)

    del X_train, X_val
    del y_train, y_val, train_probs, val_probs
    gc.collect()

print("Done")
for k in list(macro_roc_auc.keys()):
    print(f"Macro roc-auc for {k}:", sum(macro_roc_auc[k]) / len(macro_roc_auc[k]))

## Понимаю, выглядит страшно. Но всё куда проще:
## Обучаем каждую модель внутри одного цикла, считаем valid_roc_auc и складываем, а так же сохраняем предикты каждой для дальнейшего обучение ансамбля.

# Дожимаем задачу: анасамбль

In [ ]:
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

optuna.logging.set_verbosity(optuna.logging.ERROR)

STACK_DIR = MODELS_DIR / "stacking_logreg"
STACK_DIR.mkdir(parents=True, exist_ok=True)

stack_meta = {}
stack_macro_roc_auc = []

model_names = ["catboost","lgbm","xgb","mlp"]

macro_roc_auc['ensemble'] = []

for y_col in y.columns.to_list()[1:]:

    print(f"Start stacking for {y_col}")
    print("=" * 30)

    train_parts = []
    val_parts = []

    for model_name in model_names:
        train_pred = pd.read_parquet(PREDS_DIR / model_name / f"train_{model_name}_{y_col}.parquet")["pred"].values
        val_pred   = pd.read_parquet(PREDS_DIR / model_name / f"val_{model_name}_{y_col}.parquet")["pred"].values
        train_parts.append(train_pred)
        val_parts.append(val_pred)

    X_train_stack = np.column_stack(train_parts)
    X_val_stack   = np.column_stack(val_parts)

    y_train = y[y_col].astype(int).iloc[indeex_for_y_cols_train[y_col]]
    y_val   = y[y_col].astype(int).iloc[indeex_for_y_cols_test[y_col]]

    n_pos = int((y_train == 1).sum())
    n_neg = int((y_train == 0).sum())

    # =========================
    # OPTUNA
    # =========================
    class_weight = {0: 1.0, 1: scale_pos_weight}
    def objective(trial):


        penalty = trial.suggest_categorical("penalty", ["l1","l2","elasticnet"])
        C = trial.suggest_categorical("C", [0.01,0.03,0.1,0.3,1.0,3.0,10.0])
        
        if penalty == "elasticnet":
            l1_ratio = trial.suggest_categorical("l1_ratio", [0.2,0.5,0.8])
        else:
            l1_ratio = None
        
        model = LogisticRegression(
            penalty=penalty,
            C=C,
            solver="saga",
            l1_ratio=l1_ratio,
            class_weight=class_weight,  # ты его уже считаешь
            max_iter=1000,
            random_state=42,
            n_jobs=1,
        )
        model.fit(X_train_stack, y_train)
        val_probs = model.predict_proba(X_val_stack)[:,1]
        score = roc_auc_score(y_val, val_probs) if y_val.nunique()>1 else 0.5

        return score

    sampler = optuna.samplers.TPESampler(n_startup_trials=8, seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=30)

    best_params = study.best_params

    # =========================
    # ФИНАЛЬНАЯ МОДЕЛЬ
    # =========================
    model = LogisticRegression(
        **best_params,
        solver="saga",
        class_weight=class_weight,
        max_iter=1000,
        random_state=42,
        n_jobs=1,
    )

    model.fit(X_train_stack, y_train)

    train_probs = model.predict_proba(X_train_stack)[:,1]
    val_probs   = model.predict_proba(X_val_stack)[:,1]

    train_roc_auc = roc_auc_score(y_train, train_probs) if y_train.nunique()>1 else 0.5
    valid_roc_auc = roc_auc_score(y_val, val_probs) if y_val.nunique()>1 else 0.5

    model_path = STACK_DIR / f"{y_col}.pkl"
    joblib.dump(model, model_path)

    pd.DataFrame({"pred": train_probs}).to_parquet(STACK_DIR / f"train_stack_{y_col}.parquet")
    pd.DataFrame({"pred": val_probs}).to_parquet(STACK_DIR / f"val_stack_{y_col}.parquet")

    stack_meta[y_col] = {
        "model_path": str(model_path),
        "train_roc_auc": float(train_roc_auc),
        "valid_roc_auc": float(valid_roc_auc),
        "best_params": best_params,
    }

    with open(STACK_DIR / "meta_model.json", "w", encoding="utf-8") as f:
        json.dump(stack_meta, f, ensure_ascii=False, indent=2)

    macro_roc_auc['ensemble'].append(valid_roc_auc)

    print(f"Stacking {y_col}: {valid_roc_auc:.6f}")
    print("=" * 30)

    del X_train_stack, X_val_stack, model, train_probs, val_probs
    gc.collect()

print("Macro ROC-AUC:", sum(macro_roc_auc['ensemble'])/len(macro_roc_auc['ensemble']))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec

data = {}
for model_name in os.listdir(str(MODELS_DIR)):
    with open(os.path.join(MODELS_DIR / model_name, "meta_model.json"), "r", encoding="utf-8") as f:
        model_meta = json.load(f)
        for targ in list(model_meta.keys()):
            roc_aucs.append(model_meta[targ]["valid_roc_auc"])
        data[model_name] = roc_aucs


data = np.array([macro_roc_auc[m] for m in models])


fig = plt.figure(figsize=(10, 6))
gs = GridSpec(2, 2, figure=fig)

# --- 1. ROC AUC по таргетам ---
ax1 = fig.add_subplot(gs[0, :])
for i, m in enumerate(models):
    ax1.plot(data[i], label=m)

ax1.set_title("ROC AUC по таргетам")
ax1.set_xticks(range(len(targets)))
ax1.set_xticklabels(targets, rotation=90, fontsize=7)
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# --- 2. Средний ROC AUC ---
ax2 = fig.add_subplot(gs[1, 0])
avg_scores = data.mean(axis=1)
ax2.bar(models, avg_scores)

ax2.set_title("Средний ROC AUC")
ax2.set_xticklabels(models, rotation=30)
ax2.grid(True, axis='y', alpha=0.3)

# --- 3. Отставание от лучшей ---
ax3 = fig.add_subplot(gs[1, 1])
best_per_target = data.max(axis=0)
gap = best_per_target - data

for i, m in enumerate(models):
    ax3.plot(gap[i], label=m)

ax3.set_title("Отставание от лучшей модели")
ax3.set_xticks(range(len(targets)))
ax3.set_xticklabels(targets, rotation=90, fontsize=7)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()